In [1]:
#%pip install datasets -q

In [2]:
#%pip installsentence-transformers transformers

In [3]:
#%pip install arabert fastembed

In [4]:
#%pip install qdrant-client langchain langchain-community langchain-qdrant

In [5]:
from datasets import load_dataset

dataset = load_dataset("khalidalt/SANAD", "default", split="train")


s:\my_projects\Arabic_News_Agentic_RAG\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
print(dataset)

Dataset({
    features: ['article', 'category'],
    num_rows: 99810
})


In [7]:
dataset[0]

{'article': '\nالمخلافي يلتقي ولد الشيخ ويقدم جدول أعمال مشاورات جنيف\n\nالحدث.نت\n\nالتقى نائب رئيس مجلس الوزراء، وزير الخارجية اليمني عبدالملك المخلافي المبعوث الأممي إسماعيل ولد الشيخ احمد، وخلال اللقاء قدم المخلافي الذي يرأس الوفد الحكومي للمفاوضات اليمنية أجندة وجدول أعمال مشاورات جنيف، بالإضافة إلى ملاحظات عامة وأخرى تفصيلية على المسودة المقترحة للمشاورات.وشدد المخلافي على ضرورة  تطبيق الميليشيات الانقلابية لقرار مجلس الأمن 2216  والقرارات ذات الصلة، مجدداً على  حرص الحكومة اليمنية على السلام.وأكد وزير الخارجية اليمني جدية السلطة الشرعية في التعاطي الايجابي مع كافة الجهود الدولية الهادفة الى تحقيق السلام وتطبيق قرارات الشرعية الدولية.\xa0\n',
 'category': 'Politics'}

In [8]:
import re

In [9]:
def clean_text(text):
    text = re.sub(r'[إأآا]', 'ا', text)
    # Remove diacritics (tashkeel)
    text = re.sub(r'[\u064B-\u065F]', '', text)
    
    text = re.sub(r'[^\u0600-\u06FF0-9\s\.\،\؟\!]', '', text)
    # Remove tatweel
    text = re.sub(r'\u0640', '', text)
    # Remove non-Arabic characters except spaces and punctuation
    text = re.sub(r'[^\u0600-\u06FF\s\.\،\؟\!]', '', text)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [10]:
sample = dataset[0]['article']
sample =sample[:200]
cleaned_sample = clean_text(sample)
after_cleaning = cleaned_sample
before_cleaning = sample

In [11]:
print("Before cleaning:")
print(before_cleaning)
print("\nAfter cleaning:")
print(after_cleaning)

Before cleaning:

المخلافي يلتقي ولد الشيخ ويقدم جدول أعمال مشاورات جنيف

الحدث.نت

التقى نائب رئيس مجلس الوزراء، وزير الخارجية اليمني عبدالملك المخلافي المبعوث الأممي إسماعيل ولد الشيخ احمد، وخلال اللقاء قدم المخلافي

After cleaning:
المخلافي يلتقي ولد الشيخ ويقدم جدول اعمال مشاورات جنيف الحدث.نت التقى نائب رئيس مجلس الوزراء، وزير الخارجية اليمني عبدالملك المخلافي المبعوث الاممي اسماعيل ولد الشيخ احمد، وخلال اللقاء قدم المخلافي


In [12]:
#! pip install qdrant-client langchain langchain-community langchain-qdrant sentence-transformers transformers datasets arabert fastembed -q

In [13]:
#! pip install langchain-text-splitters -q

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=['\n\n', '\n', ' ']
)

In [15]:
def prepare_documents(dataset, max_articles=5000):
    docs = []
    for i, item in enumerate(dataset):
        if i >= max_articles:
            break
        text = clean_text(item['article'])
        if len(text) < 50:
            continue
        chunks = text_splitter.split_text(text)
        for j, chunk in enumerate(chunks):
            chunk = chunk.strip()
            if len(chunk) < 20:
                continue
            docs.append({
                'id': f"{i}_{j}",
                'text': chunk,
                'category': item.get('category', 'unknown'),
                'source_idx': i
            })
    return docs
docs = prepare_documents(dataset)
print(f"total chunks: {len(docs)}")
print(docs[0])

total chunks: 26320
{'id': '0_0', 'text': 'المخلافي يلتقي ولد الشيخ ويقدم جدول اعمال مشاورات جنيف الحدث.نت التقى نائب رئيس مجلس الوزراء، وزير الخارجية اليمني عبدالملك المخلافي المبعوث الاممي اسماعيل ولد الشيخ احمد، وخلال اللقاء قدم المخلافي الذي يراس الوفد الحكومي للمفاوضات اليمنية اجندة وجدول اعمال مشاورات جنيف، بالاضافة الى ملاحظات عامة', 'category': 'Politics', 'source_idx': 0}


In [16]:
print(dataset.column_names)
print(dataset[0])

['article', 'category']
{'article': '\nالمخلافي يلتقي ولد الشيخ ويقدم جدول أعمال مشاورات جنيف\n\nالحدث.نت\n\nالتقى نائب رئيس مجلس الوزراء، وزير الخارجية اليمني عبدالملك المخلافي المبعوث الأممي إسماعيل ولد الشيخ احمد، وخلال اللقاء قدم المخلافي الذي يرأس الوفد الحكومي للمفاوضات اليمنية أجندة وجدول أعمال مشاورات جنيف، بالإضافة إلى ملاحظات عامة وأخرى تفصيلية على المسودة المقترحة للمشاورات.وشدد المخلافي على ضرورة  تطبيق الميليشيات الانقلابية لقرار مجلس الأمن 2216  والقرارات ذات الصلة، مجدداً على  حرص الحكومة اليمنية على السلام.وأكد وزير الخارجية اليمني جدية السلطة الشرعية في التعاطي الايجابي مع كافة الجهود الدولية الهادفة الى تحقيق السلام وتطبيق قرارات الشرعية الدولية.\xa0\n', 'category': 'Politics'}


In [17]:
#%pip install sentence-transformers -q

In [ ]:
#%pip install langchain-community -q

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (Distance, VectorParams, SparseVectorParams, SparseIndexParams)
from langchain_community.embeddings import HuggingFaceEmbeddings

C:\Users\saleen\AppData\Local\Temp\ipykernel_14552\2704648671.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [ ]:
client = QdrantClient(path="./qdrant_db")

In [ ]:
#! pip install sentence-transformers

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="aubmindlab/bert-base-arabertv02",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

C:\Users\saleen\AppData\Local\Temp\ipykernel_14552\4233599945.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3385.63it/s]
[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.tra

In [ ]:
test = embeddings.embed_query("ماذا حدث؟")
print(test)

[0.0363709032535553, 0.04516449198126793, -0.019641391932964325, -0.024854125455021858, 0.010806656442582607, 0.010313255712389946, -0.0458824522793293, 0.018845660611987114, -0.05937166139483452, -0.0074544488452374935, -0.009409819729626179, -0.006403186358511448, -0.03919430077075958, 0.0031174225732684135, 0.005650725681334734, -0.009808111935853958, 0.003228807356208563, -0.031219787895679474, -0.04851827770471573, 0.007481314707547426, 0.030651241540908813, -0.0019851333927363157, 0.019228190183639526, -0.05994177982211113, -0.014437217265367508, 0.00585546437650919, 0.03899255394935608, 0.02101915515959263, -0.011374888010323048, -0.04259997978806496, -0.10116146504878998, 0.01992419734597206, 0.031495969742536545, 0.0002917976526077837, -0.0022383693140000105, -0.02202344313263893, 0.046926431357860565, 0.025013089179992676, 0.042713504284620285, -0.009916973300278187, -0.013580034486949444, 0.008750037290155888, -0.013812736608088017, -0.0061036464758217335, -0.000778508721850

In [28]:
collection_name = "arabic_news"

# Delete if exists (clean slate)
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=768,
        distance=Distance.COSINE
    ),
    sparse_vectors_config={
        "sparse": SparseVectorParams(
            index=SparseIndexParams(on_disk=False)
        )
    }
)

print(f"Collection '{collection_name}' created successfully")

Collection 'arabic_news' created successfully


In [29]:
print(client.get_collection(collection_name))

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=0 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=768, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, sparse_vectors={'sparse': SparseVectorParams(index=SparseIndexParams(full_scan_threshold=None, on_disk=False, datatype=None), modifier=None)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=No

In [30]:
from qdrant_client.models import PointStruct , SparseVector
from fastembed import SparseTextEmbedding
from langchain_community.embeddings import HuggingFaceEmbeddings
import uuid

In [31]:
embeddings = HuggingFaceEmbeddings(
    model_name="aubmindlab/bert-base-arabertv02",
    model_kwargs={"device":"cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2239.43it/s]
[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")

In [33]:
import hashlib

def make_point_id(doc):
    unique_string = f"{doc['source_idx']}_{doc['id']}"
    return hashlib.md5(unique_string.encode()).hexdigest()

In [34]:
def index_documents(docs, batch_size=64):
    total=len(docs)
    for i in range(0, total, batch_size):
        batch = docs[i:i+batch_size]
        texts= [d['text']for d in batch]

        dense_vecs = embeddings.embed_documents(texts)
        sparse_vecs = list(sparse_model.embed(texts))

        points = []
        for j, doc in enumerate(batch):
            points.append(PointStruct(
                id=make_point_id(doc),
                vector={
                "":dense_vecs[j],
                "sparse": SparseVector(
                    indices=sparse_vecs[j].indices.tolist(),
                    values=sparse_vecs[j].values.tolist()
                )
                },
                payload={
                    "text": doc['text'],
                    "category": doc['category'],
                    "source_idx": doc['source_idx']
                }
            ))
        client.upsert(
            collection_name=collection_name,
            points=points
        )
        if i % 640 ==0:
            print(f"Indexed {min(i+batch_size, total)}/{total} documents.")
index_documents(docs)
print('done') 

Indexed 64/26320 documents.
Indexed 704/26320 documents.
Indexed 1344/26320 documents.
Indexed 1984/26320 documents.
Indexed 2624/26320 documents.
Indexed 3264/26320 documents.
Indexed 3904/26320 documents.
Indexed 4544/26320 documents.
Indexed 5184/26320 documents.
Indexed 5824/26320 documents.
Indexed 6464/26320 documents.
Indexed 7104/26320 documents.
Indexed 7744/26320 documents.
Indexed 8384/26320 documents.
Indexed 9024/26320 documents.
Indexed 9664/26320 documents.
Indexed 10304/26320 documents.
Indexed 10944/26320 documents.
Indexed 11584/26320 documents.
Indexed 12224/26320 documents.
Indexed 12864/26320 documents.
Indexed 13504/26320 documents.
Indexed 14144/26320 documents.
Indexed 14784/26320 documents.
Indexed 15424/26320 documents.
Indexed 16064/26320 documents.
Indexed 16704/26320 documents.
Indexed 17344/26320 documents.
Indexed 17984/26320 documents.
Indexed 18624/26320 documents.
Indexed 19264/26320 documents.
Indexed 19904/26320 documents.


C:\Users\saleen\AppData\Local\Temp\ipykernel_14552\2492321932.py:27: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20032 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(


Indexed 20544/26320 documents.
Indexed 21184/26320 documents.
Indexed 21824/26320 documents.
Indexed 22464/26320 documents.
Indexed 23104/26320 documents.
Indexed 23744/26320 documents.
Indexed 24384/26320 documents.
Indexed 25024/26320 documents.
Indexed 25664/26320 documents.
Indexed 26304/26320 documents.
done
